<a href="https://colab.research.google.com/github/Amalmohamed64/tech-tasks/blob/main/task1_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A — Data Understanding and Regex
## Task 1 — Load and Explore the CSV

In [ ]:
import os
import glob
import pandas as pd
from google.colab import files

# Look for any uploaded CSV file containing 'nlp_customer_messages'
csv_files = glob.glob('*nlp_customer_messages*.csv')

if csv_files:
    # Choose the most recent or main uploaded file
    file_path = csv_files[-1]
else:
    file_path = '/content/nlp_customer_messages/nlp_customer_messages.csv'

# Fallback to direct upload if the expected path does not exist
if not os.path.exists(file_path):
    print(f"File not found. Please upload your CSV file:")
    uploaded = files.upload()
    uploaded_files = list(uploaded.keys())
    if uploaded_files:
        file_path = uploaded_files[0]

# Load the dataset
df = pd.read_csv(file_path)
print(f"Dataset successfully loaded from: {file_path}")

Dataset successfully loaded from: nlp_customer_messages - nlp_customer_messages (3).csv


### 1. Initial Dataset Exploration
We display the first 5 rows, check the dimensions of the dataset, and inspect the column names to understand its structure.

In [ ]:
# 1. Display first 5 rows
print("--- First 5 Rows of the Dataset ---")
display(df.head())

# 2. Display dataset shape
print(f"\nDataset Shape: {df.shape} (rows, columns)")

# 3. Display column names
print("\nColumn Names:")
print(list(df.columns))

--- First 5 Rows of the Dataset ---


,message_id,message
0,MSG001,"Good morning, My phone is not working :( Than..."
1,MSG002,"Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive..."



Dataset Shape: (80, 2) (rows, columns)

Column Names:
['message_id', 'message']


### 2. Missing Values Analysis
We check for missing values across all columns to evaluate data completeness.

In [ ]:
# Calculate missing values in every column
missing_counts = df.isnull().sum()
missing_table = pd.DataFrame({
    'Column Name': missing_counts.index,
    'Missing Values Count': missing_counts.values
})

print("--- Missing Values Analysis ---")
display(missing_table)

# Mention whether the dataset contains missing values
total_missing = missing_counts.sum()
if total_missing > 0:
    print(f"The dataset contains missing values. Affected columns and counts are shown above.")
else:
    print("The dataset does not contain any missing values.")

--- Missing Values Analysis ---


,Column Name,Missing Values Count
0,message_id,0
1,message,0


The dataset does not contain any missing values.


### 3. Duplicate Message IDs
We verify if the `message_id` column contains any duplicate identifier records.

In [ ]:
if 'message_id' in df.columns:
    duplicates = df[df.duplicated(subset=['message_id'], keep=False)]
    num_duplicates = df.duplicated(subset=['message_id']).sum()
    print(f"Total duplicated message IDs: {num_duplicates}")
    if num_duplicates > 0:
        print("\nDuplicated records found:")
        display(duplicates.sort_values(by='message_id'))
    else:
        print("No duplicate message IDs found. All records have a unique ID.")
else:
    print("Column 'message_id' is not present in the dataset.")

Total duplicated message IDs: 0
No duplicate message IDs found. All records have a unique ID.


### 4. Extreme Message Lengths
Using the text column (typically `message_text` or `text`), we identify the longest and shortest messages by character count.

In [ ]:
# Identify the actual text column
text_col = 'message' if 'message' in df.columns else ('message_text' if 'message_text' in df.columns else ('text' if 'text' in df.columns else None))

if text_col:
    # Temporarily calculate character lengths safely
    lengths = df[text_col].fillna("").str.len()

    max_len = lengths.max()
    min_len = lengths.min()

    # Get all indices that share the maximum length (handling ties)
    longest_indices = lengths[lengths == max_len].index.tolist()
    # Get all indices that share the minimum length (handling ties)
    shortest_indices = lengths[lengths == min_len].index.tolist()

    print(f"--- Longest Messages (Length: {max_len} characters) ---")
    for idx in longest_indices:
        print(f"Index {idx}:")
        print(df.loc[idx, text_col])
        print()

    print("-" * 50)

    print(f"--- Shortest Messages (Length: {min_len} characters) ---")
    for idx in shortest_indices:
        print(f"Index {idx}:")
        print(df.loc[idx, text_col])
        print()
else:
    print("Could not find a standard text/message column.")

--- Longest Messages (Length: 209 characters) ---
Index 35:
Hey there, I need help with payment for order 10612. Please check. Email: customer36@example.com Visit https://help.example.com/ticket/10612   Please   help   me.   My backup email is user.name+test@sample.org

--------------------------------------------------
--- Shortest Messages (Length: 46 characters) ---
Index 28:
Dear team, Thanks! Great service :) Thank you.



### 7. Save and Download Updated Dataset
We export the updated DataFrame to a CSV file named `nlp_customer_messages_updated.csv` without including the index, and trigger an automatic Colab download.

In [ ]:
if text_col:
    # Calculate length while preserving missing values correctly
    df['char_count'] = df[text_col].apply(lambda x: len(str(x)) if pd.notnull(x) else None)

    print("Sample of dataset with the new 'char_count' feature:")
    display(df[[text_col, 'char_count']].head())

    # Save the updated DataFrame to CSV
    output_filename = 'nlp_customer_messages_updated.csv'
    df.to_csv(output_filename, index=False)
    print(f"Dataset successfully saved to: {output_filename}")

    # Trigger Google Colab file download
    try:
        files.download(output_filename)
        print("Download window opened.")
    except Exception as e:
        print("Automatic download not supported in this environment, but the file is saved locally.")
else:
    print("Text column missing. Cannot compute lengths or save file.")

Sample of dataset with the new 'char_count' feature:


,message,char_count
0,"Good morning, My phone is not working :( Than...",55
1,"Hi, Thanks for your quick response! I need hel...",55
2,"Hi support, I LOVE this product 😍😍 Can someone...",84
3,"Hey there, The new update is AMAZING!! Please ...",96
4,"Dear team, The price is 250 EGP, too expensive...",81


Dataset successfully saved to: nlp_customer_messages_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download window opened.


### 6. Data Quality Summary & Explanation

#### Why message length is useful during NLP data exploration:
- **Outlier Detection**: Extremely long or short messages can highlight anomalous entries (e.g. empty strings, automated system logs, spam, or copy-pasted templates).
- **Preprocessing Decisions**: Message lengths help us choose a token threshold (sequence truncation or padding limits) when building deep learning models.
- **Feature Engineering**: Longer customer messages might contain more complex sentiment, multiple entities, or represent a higher level of frustration/detail than short ones.

#### Final Summary of Data Quality:
- **Total Rows and Columns**: 80 rows and 3 columns (including the newly created `char_count`).
- **Missing Values**: 0 missing values across all columns.
- **Duplicated Message IDs**: 0 duplicates found in the `message_id` column.
- **Longest Message**: 209 characters (Index 35).
- **Shortest Message**: 46 characters (Index 28).
- **`char_count` status**: Successfully created as a stable integer column without converting actual nulls (if any) to string "nan" types.

## Task 2 — Regex: Find and Extract

In this section, we use Python's built-in `re` module to locate, parse, and extract structural information (emails, URLs, specific phone formats, digit groups, and case-insensitive word matching) from the unstructured message text. This helps enrich our NLP dataset with structured metadata.

In [ ]:
import re

# Identify actual text column dynamically from Task 1
text_col = 'message' if 'message' in df.columns else ('message_text' if 'message_text' in df.columns else ('text' if 'text' in df.columns else None))
print(f"Target text column identified for extraction: '{text_col}'")

Target text column identified for extraction: 'message'


### Step 2 — Extract Email Addresses
We extract email addresses using a pattern that looks for alphanumeric characters and common symbols, followed by `@`, and a domain suffix.

* **Regex Pattern**: `[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}`
* **Why it fits**: It matches standard email syntax while safely ignoring surrounding text and punctuation.

In [ ]:
email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

# Safe extraction function using re.findall
def extract_emails(text):
    if pd.isnull(text):
        return []
    return re.findall(email_pattern, str(text))

df['emails'] = df[text_col].apply(extract_emails)
print("Emails column successfully created.")

Emails column successfully created.


### Step 3 — Extract URLs
We extract links starting with `http://`, `https://`, or `www.`.

* **Regex Pattern**: `(?:https?://|www\.)\S+` (with trailing punctuation stripped afterwards to clean up results).
* **Why it fits**: This pattern captures standard protocols and subdomain declarations accurately without getting bogged down by domain structures.

In [ ]:
url_pattern = r'(?:https?://|www\.)\S+'

def extract_urls(text):
    if pd.isnull(text):
        return []
    found = re.findall(url_pattern, str(text))
    # Strip common trailing punctuation (like periods or commas) from URLs
    cleaned = [link.rstrip('.,;)!?') for link in found]
    return cleaned

df['urls'] = df[text_col].apply(extract_urls)
print("URLs column successfully created.")

URLs column successfully created.


### Step 4 — Extract Phone Numbers
We extract mobile numbers that start specifically with `010` and are followed by exactly 8 digits.

* **Regex Pattern**: `\b010\d{8}\b`
* **Why it fits**: The word boundary `\b` guarantees that we do not mistakenly match parts of longer numeric digits, securing exactly an 11-digit structure.

In [ ]:
phone_pattern = r'\b010\d{8}\b'

def extract_phones(text):
    if pd.isnull(text):
        return []
    return re.findall(phone_pattern, str(text))

df['phones'] = df[text_col].apply(extract_phones)
print("Phones column successfully created.")

Phones column successfully created.


### Step 5 — Extract Number Groups
We capture all instances of consecutive digits.

* **Regex Pattern**: `\d+`
* **Why it fits**: Matches sequence structures like order IDs, tickets, pricing, and timestamps as strings to maintain any leading zeros.

In [ ]:
number_pattern = r'\d+'

def extract_numbers(text):
    if pd.isnull(text):
        return []
    return re.findall(number_pattern, str(text))

df['numbers'] = df[text_col].apply(extract_numbers)
print("Numbers column successfully created.")

Numbers column successfully created.


### Step 6 — Detect Urgent Messages
We inspect whether the term `urgent` is specifically mentioned as an individual word.

* **Regex Pattern**: `\burgent\b` combined with the `re.IGNORECASE` flag.
* **Why it fits**: Word boundaries restrict matches to the complete word, preventing false positives on words like "urgently" or "nonurgent" while being fully case-insensitive.

In [ ]:
urgent_pattern = r'\burgent\b'

def detect_urgent(text):
    if pd.isnull(text):
        return False
    return bool(re.search(urgent_pattern, str(text), re.IGNORECASE))

df['has_urgent'] = df[text_col].apply(detect_urgent)
print("has_urgent column successfully created.")

has_urgent column successfully created.


### Step 7 — Display and Verify Results
Let us check the head of the updated DataFrame and present aggregate metrics of the extracted features.

In [ ]:
# 1. Display first 5 rows with the newly added extraction columns
columns_to_show = [text_col, 'emails', 'urls', 'phones', 'numbers', 'has_urgent']
print("--- First 5 Rows of Extraction Results ---")
display(df[columns_to_show].head())

# 2. Display examples with non-empty results where possible
print("\n--- Sample Matches: Emails ---")
display(df[df['emails'].apply(len) > 0][columns_to_show].head(2))

print("\n--- Sample Matches: URLs ---")
display(df[df['urls'].apply(len) > 0][columns_to_show].head(2))

print("\n--- Sample Matches: Phones ---")
display(df[df['phones'].apply(len) > 0][columns_to_show].head(2))

print("\n--- Sample Matches: Urgent Messages ---")
display(df[df['has_urgent'] == True][columns_to_show].head(2))

--- First 5 Rows of Extraction Results ---


,message,emails,urls,phones,numbers,has_urgent
0,"Good morning, My phone is not working :( Than...",[],[],[],[],False
1,"Hi, Thanks for your quick response! I need hel...",[],[],[],[],False
2,"Hi support, I LOVE this product 😍😍 Can someone...",[customer3@example.com],[],[],[3],False
3,"Hey there, The new update is AMAZING!! Please ...",[],[https://help.example.com/ticket/10068],[],[10068],False
4,"Dear team, The price is 250 EGP, too expensive...",[],[],[01010011705],"[250, 01010011705]",False



--- Sample Matches: Emails ---


,message,emails,urls,phones,numbers,has_urgent
2,"Hi support, I LOVE this product 😍😍 Can someone...",[customer3@example.com],[],[],[3],False
5,"Dear support, The package for order 10102 is d...",[customer6@example.com],[],[],"[10102, 6]",False



--- Sample Matches: URLs ---


,message,emails,urls,phones,numbers,has_urgent
3,"Hey there, The new update is AMAZING!! Please ...",[],[https://help.example.com/ticket/10068],[],[10068],False
7,"Hello team, Great quality, but delivery took 7...",[],[https://help.example.com/ticket/10136],[],"[7, 10136]",False



--- Sample Matches: Phones ---


,message,emails,urls,phones,numbers,has_urgent
4,"Dear team, The price is 250 EGP, too expensive...",[],[],[01010011705],"[250, 01010011705]",False
9,"Hi, Can you change my delivery address? I need...",[],[],[01010023410],[01010023410],False



--- Sample Matches: Urgent Messages ---


,message,emails,urls,phones,numbers,has_urgent
6,"Hello, Bad quality... very disappointed. What ...",[],[],[],[],True
13,"Dear support, The app crashes every time I ope...",[],[],[],[],True


In [ ]:
# 3. Show Summary of Extraction Statistics
summary_stats = {
    'Total messages containing emails': (df['emails'].apply(len) > 0).sum(),
    'Total messages containing URLs': (df['urls'].apply(len) > 0).sum(),
    'Total messages containing phone numbers': (df['phones'].apply(len) > 0).sum(),
    'Total messages containing consecutive digit groups': (df['numbers'].apply(len) > 0).sum(),
    'Total messages marked as urgent': df['has_urgent'].sum()
}

for metric, count in summary_stats.items():
    print(f"{metric}: {count}")

# 4. Verify columns exist
print(f"\nAll requested columns added: {all(col in df.columns for col in ['emails', 'urls', 'phones', 'numbers', 'has_urgent'])}")

Total messages containing emails: 26
Total messages containing URLs: 20
Total messages containing phone numbers: 16
Total messages containing consecutive digit groups: 58
Total messages marked as urgent: 11

All requested columns added: True


### Step 8 — Save and Export Complete Dataset
We export the combined Task 1 and Task 2 dataset and present a download option.

In [ ]:
final_output_filename = 'nlp_customer_messages_updated.csv'
df.to_csv(final_output_filename, index=False)
print(f"Final updated DataFrame containing both Task 1 and Task 2 saved as: {final_output_filename}")

try:
    files.download(final_output_filename)
    print("Download of final file initiated.")
except Exception as e:
    print("Download window could not be opened automatically, but the file is saved locally.")

Final updated DataFrame containing both Task 1 and Task 2 saved as: nlp_customer_messages_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download of final file initiated.
